# Alpha diversity

Evolved-minus-ancestral change in Shannon Index across the landscape, dose-level contrasts
and a global test of the Shannon Index landscape.

Outputs: Figure 3, Figures S5-S6, Tables S2-S3.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.spatial.distance import braycurtis

%run 00_setup_and_functions.ipynb

## Condition-level history effects
For every carbon x streptomycin condition: change in mean Shannon Index and in mean
*A. caviae* relative abundance, and the Bray-Curtis distance between the mean compositions
of the two histories (Figures 3A,B and S6).

In [2]:
rows = []
for richness, anc, evo in RICHNESS_PAIRS:
    for carbon in CARBON_VALUES:
        for strep in STREPTOMYCIN_VALUES:
            sel = np.isclose(sample_compositions.carbon_g_L, carbon) & (sample_compositions.streptomycin == strep)
            a = sample_compositions.loc[(sample_compositions.community == anc) & sel]
            e = sample_compositions.loc[(sample_compositions.community == evo) & sel]
            rows.append({"richness": richness, "carbon_g_L": carbon, "streptomycin": strep,
                         "delta_shannon": e.shannon.mean() - a.shannon.mean(),
                         "delta_1972": e.p1972.mean() - a.p1972.mean(),
                         "bray_curtis_between_histories": braycurtis(a[SPECIES].mean().to_numpy(float),
                                                                     e[SPECIES].mean().to_numpy(float))})
condition_effects = pd.DataFrame(rows)

## Table S2: dose-level contrasts
Shannon Index averaged across carbon within each landscape; Holm correction across the four doses.

In [3]:
rep_dose = sample_compositions.groupby(["community", "replicate", "streptomycin"])["shannon"].mean().reset_index()
rows = []
for richness, anc, evo in RICHNESS_PAIRS:
    family = []
    for strep in STREPTOMYCIN_VALUES:
        x = rep_dose.loc[(rep_dose.community == anc) & (rep_dose.streptomycin == strep), "shannon"].to_numpy(float)
        y = rep_dose.loc[(rep_dose.community == evo) & (rep_dose.streptomycin == strep), "shannon"].to_numpy(float)
        diff, p = exact_mean_difference_test(x, y)
        lo, hi = bootstrap_diff_ci(x, y, seed=RANDOM_SEED + int(strep) + int(richness))
        family.append({"richness": richness, "streptomycin": strep, "ancestral_mean": x.mean(), "evolved_mean": y.mean(),
                       "delta_shannon": diff, "ci95_low": lo, "ci95_high": hi, "p_exact": p})
    for r, q in zip(family, holm([r["p_exact"] for r in family])):
        r["p_holm_across_doses"] = q
    rows += family
table_s2 = pd.DataFrame(rows)
table_s2.to_csv(TABLE_DIR / "Table_S2_alpha_diversity_dose.csv", index=False)
table_s2

,richness,streptomycin,ancestral_mean,evolved_mean,delta_shannon,ci95_low,ci95_high,p_exact,p_holm_across_doses
0,3,0.0,0.552601,0.411579,-0.141021,-0.154508,-0.126362,0.000155,0.000622
1,3,2.0,0.630348,0.502821,-0.127526,-0.144376,-0.110799,0.000155,0.000622
2,3,20.0,0.453285,0.714308,0.261022,0.239186,0.282308,0.000155,0.000622
3,3,200.0,0.254905,0.704664,0.449759,0.425670,0.474526,0.000155,0.000622
4,16,0.0,0.849675,0.681546,-0.168130,-0.196002,-0.139435,0.000155,0.000622
5,16,2.0,0.822527,0.700502,-0.122025,-0.143322,-0.100611,0.000155,0.000622
6,16,20.0,0.748308,0.801815,0.053507,0.024496,0.083087,0.005594,0.005594
7,16,200.0,0.785024,0.693745,-0.091278,-0.119995,-0.058473,0.000622,0.001243


## Table S3: global test of the Shannon Index landscape
Each landscape is the vector of its 64 Shannon Index values; exact two-group PERMANOVA over
all 12,870 allocations.

In [4]:
rows = []
for richness, anc, evo in RICHNESS_PAIRS:
    sub = sample_compositions.loc[sample_compositions.community.isin([anc, evo])]
    X, labels = [], []
    for (c, r), d in sub.groupby(["community", "replicate"]):
        X.append(d.sort_values(["carbon_g_L", "streptomycin"]).shannon.to_numpy(float))
        labels.append(int(c == evo))
    F, R2, p, n_alloc = exact_permanova(np.asarray(X), np.asarray(labels))
    rows.append({"richness": int(richness), "pseudo_F": F, "R2": R2, "exact_P": p, "n_permutations": n_alloc,
                 "n_landscapes_per_history": 8})
table_s3 = pd.DataFrame(rows)
table_s3.to_csv(TABLE_DIR / "Table_S3_alpha_diversity_landscape_PERMANOVA.csv", index=False)
table_s3

,richness,pseudo_F,R2,exact_P,n_permutations,n_landscapes_per_history
0,3,59.054017,0.808361,0.000155,12870,8
1,16,37.218117,0.726659,0.000155,12870,8


## Figure 3
A,B, evolved-minus-ancestral change in Shannon Index in every environment; C, Shannon Index averaged across carbon at each dose, with replicate landscapes, means and 95% CIs (Table S2).

In [5]:
max_abs_H = float(np.nanmax(np.abs(condition_effects.delta_shannon)))
lim_H = np.ceil(max_abs_H * 10) / 10
fig = plt.figure(figsize=(183 * MM, 80 * MM))
bottom, height = 0.15, 0.71

# A, B: evolved-minus-ancestral change in Shannon Index in every condition. The diverging map
# uses the two history colours as its poles, as for other evolved-minus-ancestral differences.
axA = fig.add_axes([0.085, bottom, 0.155, height])
axB = fig.add_axes([0.255, bottom, 0.155, height], sharey=axA)
for ax, richness, letter in [(axA, "3", "A"), (axB, "16", "B")]:
    grid = (condition_effects.loc[condition_effects.richness == richness]
            .pivot(index="carbon_g_L", columns="streptomycin", values="delta_shannon")
            .reindex(index=CARBON_VALUES, columns=STREPTOMYCIN_VALUES).to_numpy(float))
    im = ax.imshow(grid, origin="upper", aspect="auto", cmap=CMAP_HISTORY, vmin=-lim_H, vmax=lim_H)
    format_grid_axis(ax, show_x=True, show_y=(ax is axA))
    ax.set_xlabel("")
    ax.set_title(f"{richness} species", fontsize=7.5, pad=4)
    panel_label(ax, letter, dx=-32 if ax is axA else -12)
axB.tick_params(axis="y", labelleft=False)
fig.text(0.2475, 0.03, STREP_AXIS_LABEL, ha="center", va="bottom", fontsize=plt.rcParams["axes.labelsize"])
cax = fig.add_axes([0.422, bottom + 0.12, 0.008, height - 0.24])
cbar = fig.colorbar(im, cax=cax, ticks=[-0.8, -0.4, 0, 0.4, 0.8])
cbar.set_label("Change in Shannon Index\n(evolved minus ancestral)", fontsize=6.2, labelpad=4)
cbar.ax.tick_params(labelsize=5.8, length=2)
cbar.outline.set_visible(False)

# C: carbon-averaged Shannon Index at each dose under each history (replicate landscapes, means and 95% CI).
axC_top = fig.add_axes([0.60, bottom + height * 0.56, 0.38, height * 0.44])
axC_bot = fig.add_axes([0.60, bottom, 0.38, height * 0.44], sharex=axC_top)
for ax, richness, anc, evo in [(axC_top, "3", "ANC_THREE", "EVO_THREE"), (axC_bot, "16", "ANC_SIXTEEN", "EVO_SIXTEEN")]:
    for hidx, (community, color) in enumerate([(anc, ANCESTRAL_COLOR), (evo, EVOLVED_COLOR)]):
        offset = 0.0
        means, sems = [], []
        for di, dose in enumerate(STREPTOMYCIN_VALUES):
            vals = rep_dose.loc[(rep_dose.community == community) & (rep_dose.streptomycin == dose), "shannon"].to_numpy(float)
            jitter = np.random.default_rng(7100 + int(richness) * 20 + di + hidx * 100).uniform(-0.05, 0.05, len(vals))
            ax.scatter(di + offset + jitter, vals, s=6, facecolor="none", edgecolor=color, lw=0.5, alpha=0.55, zorder=1)
            means.append(vals.mean()); sems.append(vals.std(ddof=1) / np.sqrt(len(vals)))
        means, sems = np.asarray(means), np.asarray(sems)
        x = np.arange(4) + offset
        ax.plot(x, means, color=color, lw=1.1, zorder=3)
        ax.errorbar(x, means, yerr=1.96 * sems, fmt="o", mfc=color, mec=color, color=color,
                    ms=4, elinewidth=1.0, capsize=0, zorder=4)
    ax.set_ylabel("Shannon Index")
    ax.set_xticks(range(4)); ax.set_xlim(-0.45, 3.45)
    ax.set_title(f"{richness} species", fontsize=7.5, loc="left", pad=3)
panel_label(axC_top, "C", dx=-36)
axC_top.tick_params(axis="x", labelbottom=False)
axC_bot.set_xticklabels([f"{x:g}" for x in STREPTOMYCIN_VALUES])
fig.text(0.79, 0.03, STREP_AXIS_LABEL, ha="center", va="bottom", fontsize=plt.rcParams["axes.labelsize"])
fig.legend(handles=[
    Line2D([0], [0], marker="o", color=ANCESTRAL_COLOR, mfc=ANCESTRAL_COLOR, mec=ANCESTRAL_COLOR, lw=1.1, ms=3.5, label=r"Ancestral $\it{A.\ caviae}$"),
    Line2D([0], [0], marker="o", color=EVOLVED_COLOR, mfc=EVOLVED_COLOR, mec=EVOLVED_COLOR, lw=1.1, ms=3.5, label=r"Resistance-evolved $\it{A.\ caviae}$")],
    loc="upper right", bbox_to_anchor=(0.985, 1.0), ncol=2, frameon=False, fontsize=6.2, handlelength=1.6, columnspacing=1.2)
save_figure(fig, "Figure_3_alpha_diversity")

## Figure S5: mean Shannon Index landscapes

In [6]:
fig = plt.figure(figsize=(183 * MM, 76 * MM))
gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 1, 1, 0.06], wspace=0.16)
mean_H = sample_compositions.groupby(["community", "carbon_g_L", "streptomycin"]).shannon.mean().reset_index()
axes = []
for col, c in enumerate(COMMUNITY_TYPES):
    ax = fig.add_subplot(gs[0, col], sharey=axes[0] if axes else None); axes.append(ax)
    grid = (mean_H.loc[mean_H.community == c].pivot(index="carbon_g_L", columns="streptomycin", values="shannon")
            .reindex(index=CARBON_VALUES, columns=STREPTOMYCIN_VALUES).to_numpy(float))
    im = ax.imshow(grid, origin="upper", aspect="auto", vmin=0, vmax=mean_H.shannon.max(), cmap=CMAP_SEQ)
    format_grid_axis(ax, show_x=True, show_y=(col == 0))
    if col > 0:
        ax.set_ylabel("")
    ax.text(0.5, 1.03, PANEL_TREATMENT_LABELS[c], transform=ax.transAxes, ha="center", va="bottom", fontsize=5.6)
fig.colorbar(im, cax=fig.add_subplot(gs[0, 4])).set_label("Shannon Index")
fig.subplots_adjust(left=0.07, right=0.96, bottom=0.20, top=0.83)
save_figure(fig, "Figure_S5_shannon_landscapes")

## Figure S6: condition-level change in *A. caviae* abundance and Bray-Curtis distance between histories

In [7]:
fig = plt.figure(figsize=(140 * MM, 150 * MM))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.07], wspace=0.24, hspace=0.28)
m = np.nanmax(np.abs(condition_effects.delta_1972))
for col, richness in enumerate(["3", "16"]):
    sub = condition_effects.loc[condition_effects.richness == richness]
    d1972 = sub.pivot(index="carbon_g_L", columns="streptomycin", values="delta_1972").reindex(index=CARBON_VALUES, columns=STREPTOMYCIN_VALUES).to_numpy(float)
    bc = sub.pivot(index="carbon_g_L", columns="streptomycin", values="bray_curtis_between_histories").reindex(index=CARBON_VALUES, columns=STREPTOMYCIN_VALUES).to_numpy(float)
    ax_top = fig.add_subplot(gs[0, col]); ax_bot = fig.add_subplot(gs[1, col], sharex=ax_top, sharey=ax_top)
    im1 = ax_top.imshow(d1972, origin="upper", aspect="auto", cmap=CMAP_HISTORY, vmin=-m, vmax=m)
    im2 = ax_bot.imshow(bc, origin="upper", aspect="auto", cmap=CMAP_SEQ, vmin=0, vmax=condition_effects.bray_curtis_between_histories.max())
    for row, ax in enumerate([ax_top, ax_bot]):
        format_grid_axis(ax, show_x=(row == 1), show_y=(col == 0))
        if col > 0:
            ax.set_ylabel("")
    ax_top.text(0.5, 1.03, RICHNESS_LABELS[richness], transform=ax_top.transAxes, ha="center", va="bottom", fontsize=6.2)
fig.text(0.018, 0.72, r"Change in $\it{Aeromonas\ caviae}$ abundance", rotation=90, va="center", fontsize=7)
fig.text(0.018, 0.29, "Bray-Curtis distance between histories", rotation=90, va="center", fontsize=7)
fig.colorbar(im1, cax=fig.add_subplot(gs[0, 2])).set_label("Resistance-evolved minus ancestral history")
fig.colorbar(im2, cax=fig.add_subplot(gs[1, 2])).set_label("Bray-Curtis distance")
fig.subplots_adjust(left=0.15, right=0.96, bottom=0.10, top=0.92)
save_figure(fig, "Figure_S6_focal_abundance_and_bray_curtis")